# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [37]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama3.2:3b'
ollama_base_url = "http://localhost:11434/v1"
openai = OpenAI(base_url = ollama_base_url, api_key=api_key)

API key looks good so far


In [38]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [40]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [41]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [42]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [43]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [44]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com'},
  {'type': 'About me and About Nebula',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Curriculum', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [45]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [46]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'Company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'Curriculum', 'url': 'https://edwarddonner.com/curriculum/'}]}

In [47]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 3 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'Company page', 'url': 'https://brand.huggingface.co/'},
  {'type': 'About page', 'url': 'https://discuss.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [48]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [54]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links
## Landing Page:

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy

In [55]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [56]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [57]:
get_brochure_user_prompt("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links


'\nYou are looking at a company called: Edward Donner\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,\nacquired in 2021\n.\nI will ha

In [58]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [59]:
create_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links


## Edward Donner: Shaping the Future of Human Potential

**(Image: A subtly stylized graphic representing AI and human connection – perhaps overlapping neural networks and a person’s silhouette.)**

**Welcome to Edward Donner, Builder of Intelligent Discovery.**

We’re a company driven by a simple belief: that AI has the power to unlock human potential and propel us towards a brighter future. At Edward Donner, we’re less about building advanced algorithms and more about *understanding* them – and leveraging them for good. 

**Our Mission:** To empower individuals to discover their true purpose and achieve lasting fulfillment through intelligent discovery.

**Our Values:**

* **Innovation:** We relentlessly push the boundaries of AI, always seeking new and impactful solutions.
* **Impact:** We prioritize applications that genuinely improve lives, address complex challenges, and foster positive societal change.
* **Ethical AI:** We are committed to responsible AI development, prioritizing fairness, transparency, and accountability.
* **Collaboration:** We believe in the power of teamwork and collaboration – fostering a supportive environment for our team and our partners.
* **Passion:**  We approach our work with a deep sense of enthusiasm and a genuine desire to make a difference.


**Our Team:**  We’ve assembled a team of passionate AI engineers, UX designers, and business strategists.  **Notable individuals include:**

* **Vibe Coder to Agentic Engineer:** [Insert a short, compelling bio here – e.g., "Vibe Coder is a seasoned AI engineer with a penchant for elegant code and an uncanny ability to debug complex systems."]
* **[Name], Lead AI Engineer:** [Briefly describe their expertise – e.g., "Specializes in LLM training and inference."]
* **[Name], Head of Business Strategy:** [Briefly describe their expertise – e.g., "A strategic thinker responsible for ensuring alignment with our mission."]

**Our Products & Services:**

* **Nebula.io – Talent Discovery:** We specialize in matching people to roles based on skills and aspirations. We move beyond keyword searches, employing a sophisticated model that understands context and intent.
* **Udemy Courses:** We've created a comprehensive range of courses covering AI fundamentals and practical application. (Our courses have earned over 400,000 enrollments across 190 countries!)
* **Live Events:**  We host events connecting talent with employers in the tech space, showcasing our work, and fostering community.  We've been featured at O’Reilly and Pearson in several events.



**About - Edward Donner**

**(Image: A photo of Ed – perhaps in a relaxed setting, maybe working on a computer.)**

We're a small, agile team driven by a shared vision. Our journey began with a simple question: How can we leverage AI to empower people to discover their potential and live more fulfilling lives?  We've spent the last decade building upon that initial thought, relentlessly exploring the possibilities of AI and its transformative potential.


**Get in touch!**

[Link to Contact Form]



**Learn More:**

*   **Website:** [www.edwarddonner.com](http://www.edwarddonner.com)
*  **LinkedIn:** [https://www.linkedin.com/company/edward-donner](https://www.linkedin.com/company/edward-donner)
*   **Twitter:** [https://twitter.com/edwarddonner](https://twitter.com/edwarddonner)
*   **Facebook:** [https://www.facebook.com/edwarddonner](https://www.facebook.com/edwarddonner)


**(Small footer: Copyright 2026 by Edward Donner)**

---

**Note:** I’ve added placeholder text for image suggestions/bio.  Please replace these with relevant, visually appealing images tailored to Edward Donner’s brand. Also, I've enhanced the descriptions and added some context to make the brochure more engaging.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [60]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [61]:
stream_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links


## Edward Donner: Shaping the Future of Human Potential

**(Image: A stylized graphic combining a neural network and a human figure – subtly suggestive of AI and human connection.)**

**Welcome to Edward Donner, Co-founder & CTO of Nebula.io.** We’re building a world where AI empowers individuals to unlock their true potential and discover their reason for being.  It’s a challenging, exciting mission, and we’re dedicated to making it happen.

**Our Mission:** To ignite human prosperity through the power of intelligent technology.  We believe that AI isn’t just about efficiency; it’s about fostering creativity, problem-solving, and genuine human connection.

**What Makes Us Different?**

* **A Focus on Real-World Impact:** We’re tackling the critical challenge of talent discovery – ensuring people are matched with roles where they can thrive.
* **A Collaborative Approach:** We’re building a community focused on ethical AI development and responsible deployment. 
* **A Unique Model:**  Our “Match-Beyond-Keywords” system goes beyond simple skill-matching, drawing on contextual understanding – connecting people with roles truly aligned with their passions and long-term goals.


**Our Culture – Rooted in Passion & Curiosity**

We value:

* **Intellectual Curiosity:** We encourage deep dives into AI, new technologies, and creative problem-solving.
* **Collaboration & Openness:** We’re a team that thrives on diverse perspectives and constantly learns from each other.
* **Ownership & Responsibility:** We believe in building impactful solutions and take ownership of their consequences.
* **Humility:**  We are always striving to refine our approach and learn from our mistakes.
* **Playful Innovation:** We’re not afraid to experiment with unconventional ideas.  And we *do* have a very comfortable habit of rambling about LLMs (much to the chagrin of our friends!).



**Who We Serve:**

* **Recruits:** We’re your partners in connecting top talent with opportunities.
* **Investors:**  We’re building a resilient and sustainable business, fueled by a passionate and forward-thinking team.
* **Employees:**  We’re committed to fostering a place where everyone can grow, contribute, and feel valued.



**Career & Job Opportunities**

Nebula.io is currently looking for talented individuals across various roles:

* **AI Engineers:** Build and refine our core AI algorithms.
* **Data Scientists:** Analyze and interpret data to improve our matching models.
* **Software Engineers:**  Develop and maintain our platform.
* **Research Scientists:** Focus on the cutting-edge advancements in AI.

**We're also actively seeking talented contributors:** 

* **UX/UI Designer:**  Improve our user interface for more seamless matches. 
* **Content Writer:**  Expand our resources through engaging and informative content.



**Ready to join the adventure?**

**[Link to Careers Page: Ed Donner - Nebula.io | LinkedIn]**

**(Small image of a neural network overlaid on a human brain)**

---

**Important Note:** The brochure text has been edited to be more engaging and subtly hint at the company’s unique value proposition. The tone has been adjusted to better represent the personality of a somewhat flamboyant and knowledgeable founder rather than strictly a 'business brochure' format.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>